In [0]:
from common_utils.logging import get_logger

logger = get_logger("customer-view")

logger.info("Creating customer SQL view")

spark.sql("""
CREATE OR REPLACE VIEW retaildataplatform.silver.vw_customers AS

WITH customer_cleaned AS (
    SELECT

        customer_id,

        -- customer_name -> lowercase -> split by comma
        LOWER(TRIM(get(split(LOWER(customer_name), ','), 1))) AS first_name,

        LOWER(TRIM(get(split(LOWER(customer_name), ','), 0))) AS last_name,

        tax_id,
        tax_code,
        state,

        -- city -> uppercase
        UPPER(city) AS city,

        -- Remove trailing .0 from postcode
        REGEXP_REPLACE(postcode, '\\\\.0$', '') AS postcode,

        street,

        -- Same as F.lit("USA")
        'USA' AS country,

        number,
        unit,
        region,
        district,
        lon,
        lat,
        ship_to_address,

        -- Unix timestamp -> datetime
        FROM_UNIXTIME(TRY_CAST(valid_from AS BIGINT)) AS valid_from,

        FROM_UNIXTIME(TRY_CAST(valid_to AS BIGINT)) AS valid_to,

        units_purchased,
        loyalty_segment,
        last_update_ts

    FROM retaildataplatform.bronze.sqlserver_customers
)

SELECT
    customer_id,
    first_name,
    last_name,
    tax_id,
    tax_code,
    state,
    city,
    postcode,
    street,
    country,
    number,
    unit,
    region,
    district,
    lon,
    lat,
    ship_to_address,
    valid_from,
    valid_to,
    units_purchased,
    loyalty_segment,
    last_update_ts

FROM customer_cleaned
""")

logger.info("Customer view created successfully")

In [0]:
%sql
select * from retaildataplatform.silver.vw_customers